In [ ]:
# This works for all geoms (that don't cross the AM), irrespective of whether they are on east or west side of the antimeridian
# This will extend functionality to support COGs that cross the AM, without making a huge world-spanning dataset.

In [41]:
# Load Fiji
import geopandas as gpd
from odc.geo.geom import Geometry

fiji_tiled_geoms = gpd.read_file(
    "/Users/wj/Projects/csdr/csdr-cloud-spatial/debug_geoms_tiled.geojson"
)

print(len(fiji_tiled_geoms))

geom_index = 0
# geom_index = 7

print(fiji_tiled_geoms["geometry"][geom_index].bounds)

fiji_tiled_geoms = [Geometry(geom) for geom in fiji_tiled_geoms["geometry"]]

geometry_4326 = fiji_tiled_geoms[geom_index].assign_crs("EPSG:4326")
geom_geojson = geometry_4326.geojson(simplify=0)["geometry"]

geom_6933 = geometry_4326.to_crs("EPSG:6933")
geom_6933_bbox = geom_6933.boundingbox

9
(-180.0, -24.3819587198891, -176.267461612063, -12.820507703401365)


In [42]:
# Search and load STAC
from pystac import ItemCollection
from rustac import search_sync as rustac_search_sync

dataset_url = "../cache/datasets/seagrass/0-0-1/dep_s2_seagrass.parquet"

items = rustac_search_sync(
    dataset_url,
    intersects=geom_geojson,
    datetime="2017",
)

print(f"Found {len(items)} items")

items = ItemCollection(items)

ok_items = []
am_items = []
for item in items:
    print(item.bbox)
    xmin, ymin, xmax, ymax = item.bbox
    if xmin == -180.0 and xmax == 180.0:
        am_items.append(item)
    else:
        ok_items.append(item)

print(f"Found {len(ok_items)} ok items and {len(am_items)} AM crossing items")

print("Ok items:")
for item in ok_items:
    print(item.id, item.bbox)

print("AM crossing items:")
for item in am_items:
    print(item.id, item.bbox)

Found 12 items
[-178.30743677626327, -20.11524267928007, -177.4450541035085, -19.298529078932667]
[-179.169819449018, -16.824114445989782, -178.30743677626327, -15.991730594839629]
[-179.169819449018, -17.65281330691017, -178.30743677626327, -16.824114445989782]
[-179.169819449018, -18.477669425954502, -178.30743677626327, -17.65281330691017]
[-179.169819449018, -19.298529078932667, -178.30743677626327, -18.477669425954502]
[-179.169819449018, -20.927664878354115, -178.30743677626327, -20.11524267928007]
[-179.169819449018, -21.73565465582795, -178.30743677626327, -20.927664878354115]
[-180.0, -15.991730594839629, 180.0, -15.15582341258985]
[-180.0, -16.824114445989782, 180.0, -15.991730594839629]
[-180.0, -17.65281330691017, 180.0, -16.824114445989782]
[-180.0, -18.477669425954502, 180.0, -17.65281330691017]
[-180.0, -19.298529078932667, 180.0, -18.477669425954502]
Found 7 ok items and 5 AM crossing items
Ok items:
dep_s2_seagrass_068_018_2017 [-178.30743677626327, -20.11524267928007,

In [43]:
# Load AM crossing items in 6933, clip to geometry bbox so we discard
# the eastern-hemisphere portion, mask to geometry, filter indicator values,
# then write a COG for inspection.
from odc.geo.cog import write_cog
from odc.stac import load

indicator = "seagrass"
value_list = [1.0]

# The whole of column 66 in the DEP Seagrass COG grid is straddling the antimeridian.

# Now to try and load these to get a count in 6933.
data_6933_clipped = load(
    # am_items, # Just AM crossing items.
    items,  # All items.
    crs="EPSG:6933",
    resolution=10,
    chunks={},
    # This 'geopolygon' is the crux. Never load a massive world-spanning dataset, just load the part that intersects the geometry.
    geopolygon=geometry_4326,  # This is very important to limit the loaded data to not cross the AM.
)
print(f"Loaded data with shape {data_6933_clipped.dims}")
assert data_6933_clipped.sizes["x"] < 1_000_000, "Error: X dimension spanning world."

write_cog(
    data_6933_clipped["seagrass"],
    "am_crossing_cogs_clipped_6933.tif",
    dtype="float32",
    nodata=0,
    overwrite=True,
)

Loaded data with shape FrozenMappingWarningOnValuesAccess({'y': 139747, 'x': 36015, 'time': 1})


PosixPath('am_crossing_cogs_clipped_6933.tif')

In [ ]:
da = data_6933_clipped[indicator].where(data_6933_clipped[indicator].isin(value_list))

count = float(da.notnull().sum().values)
print(f"Counted {count} pixels with value in {value_list}")
one_pixel_area_m2 = abs(
    data_6933_clipped.odc.geobox.resolution.x
    * data_6933_clipped.odc.geobox.resolution.y
)
area_m2 = round(count * one_pixel_area_m2, 2)
print(f"{area_m2:.2f} m²")

Counted 258172.0 pixels with value in [1.0]
25817200.0
